# Phase 8 — probability calibration and final holdout evaluation

This phase compares the three fixed classifiers with sigmoid-calibrated versions. Calibration is developed using nested, match-grouped folds from the training partition. The candidate choice is frozen before the script reads any held-out test features or labels. The held-out test evaluation follows that freeze; rerunning it reproduces the same locked evaluation.

Run every cell from top to bottom. The models and settings were declared in the protocol before fitting; this notebook does not tune a threshold or model hyperparameters. Shot rows from one match can be related, so uncertainty summaries treat the match grouping seriously and do not claim independent-shot confidence intervals.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / '.venv').is_dir() and (p / 'README.md').is_file()), None)
if project_root is None:
    raise RuntimeError('Could not find the project root containing .venv/ and README.md. Open this notebook from the project checkout.')
expected_python = (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
if Path(sys.prefix).resolve() != (project_root / '.venv').resolve():
    raise RuntimeError(f'Select the project .venv kernel in VS Code. Active environment: {sys.prefix}')
if not expected_python.exists():
    raise RuntimeError(f'Project Python was not found: {expected_python}')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'Project: {project_root}')
print(f'Kernel:  {sys.executable}')

## 1. Develop calibration and freeze the training-only choice

This command fits nested sigmoid calibrators and records a locked training-only selection. It does not read test features or labels. If a prior selection exists, the script checks that it exactly matches the reproducible candidate and stops if it has changed. It never silently replaces a frozen decision. The script writes CSV tables, JSON provenance reports, model files, and figures under the project.

In [ ]:
import subprocess

script = project_root / 'scripts' / 'calibrate_evaluate.py'
stage = 'develop'
print(f'Running Phase 8 stage: {stage}')
result = subprocess.run([sys.executable, str(script), '--stage', stage], cwd=project_root, check=False, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Phase 8 {stage} failed with exit code {result.returncode}; see the output above.')

## 2. Read training-only development results

Nested out-of-fold (OOF) results use each training shot for validation once, with all shots from a match held together. The calibration learner itself uses inner grouped folds, separate from the outer validation match groups. The ten-fold table is useful for seeing variation across folds; its standard deviation is descriptive and is not a confidence interval. The training-resubstitution table evaluates fitted models on the same rows used to fit them, so it is an optimistic diagnostic rather than validation performance.

In [ ]:
import pandas as pd
from IPython.display import Image, display

tables = project_root / 'reports' / 'tables'
development = pd.read_csv(tables / 'phase08_development_metrics.csv', float_precision='round_trip')
training = pd.read_csv(tables / 'phase08_training_metrics.csv', float_precision='round_trip')
folds = pd.read_csv(tables / 'phase08_fold_metrics.csv', float_precision='round_trip')
display(development[['variant', 'shot_count', 'goal_count', 'log_loss', 'brier_score', 'average_precision', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'family_selected', 'global_candidate']])
print('Training-set resubstitution metrics (optimistic; not validation):')
display(training[['variant', 'log_loss', 'brier_score', 'accuracy', 'precision', 'recall', 'f1', 'interpretation']])
print('Per-fold training-only results; standard deviations below are descriptive, not confidence intervals:')
with pd.option_context('display.max_rows', 70, 'display.max_columns', None):
    display(folds[['variant', 'fold', 'log_loss', 'brier_score', 'accuracy', 'precision', 'recall']])
fold_summary = folds.groupby('variant').agg(log_loss_mean=('log_loss', 'mean'), log_loss_std=('log_loss', 'std'), accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std')).reset_index()
display(fold_summary)
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase08_development_reliability.png')))
selection = __import__('json').loads((project_root / 'docs' / 'phase_08_selection.json').read_text(encoding='utf-8'))
print('Frozen family choices (based on training-only nested OOF log loss):')
display(pd.DataFrame([{'family': name, 'selected_variant': variant, 'raw_oof_log_loss': selection['family_development_scores'][name]['raw_oof_log_loss'], 'sigmoid_oof_log_loss': selection['family_development_scores'][name]['sigmoid_oof_log_loss']} for name, variant in selection['family_choices'].items()]))
print('Global development candidate:', selection['global_candidate'])

In [ ]:
stage = 'evaluate'
print(f'Running Phase 8 stage: {stage}')
result = subprocess.run([sys.executable, str(script), '--stage', stage], cwd=project_root, check=False, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Phase 8 {stage} failed with exit code {result.returncode}; see the output above.')

## 3. Check the locked held-out test results

These metrics describe the previously locked test matches. The raw and calibrated models are all shown, along with the predeclared prior-probability baseline. The selected variant labels were determined from training OOF log loss before test rows were read. Do not change the selection after looking at the test table; doing that would turn the test set into another development set. Rerunning this stage reproduces the locked evaluation.

In [ ]:
test_metrics = pd.read_csv(tables / 'phase08_test_metrics.csv', float_precision='round_trip')
print(f"Locked test shots: {int(test_metrics.shot_count.iloc[0])}; goals: {int(test_metrics.goal_count.iloc[0])}; non-goals: {int(test_metrics.non_goal_count.iloc[0])}")
print('Probability quality and ranking:')
display(test_metrics[['variant', 'log_loss', 'brier_score', 'average_precision', 'roc_auc', 'family_selected', 'global_candidate']])
print('Fixed-threshold classification results (threshold = 0.5):')
display(test_metrics[['variant', 'accuracy', 'precision', 'recall', 'f1', 'tn', 'fp', 'fn', 'tp']])
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase08_test_reliability.png')))
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase08_test_confusion_matrices.png')))
test_bins = pd.read_csv(tables / 'phase08_test_reliability_bins.csv', float_precision='round_trip')
with pd.option_context('display.max_rows', 70, 'display.max_columns', None):
    display(test_bins[['variant', 'bin_index', 'bin_lower', 'bin_upper', 'shot_count', 'goal_count', 'mean_probability', 'observed_goal_rate']])

## 4. Interpret the results carefully

Log loss and Brier score evaluate probability quality; lower values are better. Average precision and ROC AUC summarize ranking, while the confusion matrices use the fixed probability threshold 0.5. The reliability figures show mean predicted probability against the observed rate in fixed bins, with a second row showing how many shots support each bin. Sparse bins make apparent calibration differences unstable.

This is one historical La Liga season. Match grouping reduces train/validation leakage, but does not make shots statistically independent. Calibration does not guarantee future calibration, and these results do not establish that the selected model will generalize to other seasons. The Monte Carlo score simulation is a later phase and should use the frozen candidate only after this evaluation is complete. Full provenance is in `docs/phase_08_development_report.json`, `docs/phase_08_test_report.json`, and `docs/phase_08_selection.json`.